# 04 — Keyword Extraction
This notebook compares two keyword extraction approaches used in ClauseGuard:

| Method | Best for | How it works |
|---|---|---|
| **TF-IDF** | Multiple clauses | Finds words that are important in one clause *relative to all others* |
| **YAKE** | A single clause | Statistical heuristics — position, frequency, co-occurrence |

**Production file:** `backend/keywords.py`

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import yake
import numpy as np

## 1. Define sample clauses

In [ ]:
clauses = [
    "The vendor agrees to provide services as described in Exhibit A. Payment shall be made within 30 days of invoice receipt.",
    "All intellectual property developed specifically for the Client under this Agreement shall become the sole and exclusive property of the Client.",
    "Either party may terminate this Agreement at any time by providing thirty (30) days' written notice to the other party."
]

## 2. TF-IDF Keyword Extraction

In [ ]:
def extract_keywords_tfidf(clauses, top_n=5):
    if len(clauses) < 2:
        return [[] for _ in clauses]

    vectorizer = TfidfVectorizer(stop_words="english", max_features=200)
    tfidf_matrix = vectorizer.fit_transform(clauses)
    feature_names = vectorizer.get_feature_names_out()

    results = []
    for row in tfidf_matrix:
        row_data = row.toarray()[0]
        top_indices = row_data.argsort()[-top_n:][::-1]
        top_words = [feature_names[i] for i in top_indices if row_data[i] > 0]
        results.append(top_words)
    return results

tfidf_results = extract_keywords_tfidf(clauses)
print("TF-IDF Keywords per clause:")
for i, keywords in enumerate(tfidf_results, 1):
    print(f"  Clause {i}: {keywords}")

## 3. YAKE Keyword Extraction

In [ ]:
def extract_keywords_yake(text, top_n=5):
    kw_extractor = yake.KeywordExtractor(top=top_n, n=2)  # up to 2-word phrases
    keywords = kw_extractor.extract_keywords(text)
    return [(kw, round(score, 4)) for kw, score in keywords]

print("YAKE keywords for each clause (keyword, score — lower is more important):\n")
for i, clause in enumerate(clauses, 1):
    yake_kws = extract_keywords_yake(clause)
    print(f"Clause {i}: {yake_kws}")